# Compare Experiments

This notebook demonstrates how to compare multiple training experiments using
ClickHouse metrics and Plotly interactive visualizations.

## Prerequisites
- ClickHouse accessible with training data
- Multiple completed training runs to compare

In [ ]:
import os
import clickhouse_connect
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Connect to ClickHouse
ch_client = clickhouse_connect.get_client(
    host=os.environ.get("CLICKHOUSE_HOST", "clickhouse.logging.svc.cluster.local"),
    port=int(os.environ.get("CLICKHOUSE_PORT", "8123")),
    database=os.environ.get("CLICKHOUSE_DB", "isaac_lab"),
)
print(f"Connected to ClickHouse: {ch_client.server_version}")

## Select Experiments to Compare

Query completed experiments for a given environment and select the ones to compare.

In [ ]:
# List completed experiments for a given environment
environment = "Isaac-Cartpole-v0"

experiments_query = f"""
SELECT
    job_id,
    job_name,
    algorithm,
    started_at,
    duration_seconds,
    max_reward,
    final_reward
FROM training_jobs
WHERE environment = '{environment}'
  AND status = 'completed'
ORDER BY final_reward DESC
LIMIT 10
"""

df_experiments = ch_client.query_df(experiments_query)
print(f"Found {len(df_experiments)} completed experiments for {environment}")
df_experiments

In [ ]:
# Select top experiments to compare (take the best 3)
job_ids = df_experiments["job_id"].head(3).tolist()
job_ids_str = ", ".join([f"'{jid}'" for jid in job_ids])

# Query learning curves for selected experiments
curves_query = f"""
SELECT
    job_id,
    iteration,
    reward_mean,
    reward_std,
    policy_loss,
    value_loss
FROM training_metrics
WHERE job_id IN ({job_ids_str})
ORDER BY job_id, iteration ASC
"""

df_curves = ch_client.query_df(curves_query)
print(f"Retrieved {len(df_curves)} metric points across {len(job_ids)} experiments")

## Interactive Comparison with Plotly

In [ ]:
if len(df_curves) > 0:
    # Reward curves comparison
    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=("Mean Reward", "Policy Loss", "Value Loss", "Reward Distribution"),
    )

    colors = px.colors.qualitative.Set1
    for i, job_id in enumerate(job_ids):
        job_data = df_curves[df_curves["job_id"] == job_id]
        color = colors[i % len(colors)]
        label = job_id[:12]

        # Mean reward
        fig.add_trace(
            go.Scatter(x=job_data["iteration"], y=job_data["reward_mean"],
                       name=label, line=dict(color=color), legendgroup=label),
            row=1, col=1,
        )

        # Policy loss
        fig.add_trace(
            go.Scatter(x=job_data["iteration"], y=job_data["policy_loss"],
                       name=label, line=dict(color=color), legendgroup=label,
                       showlegend=False),
            row=1, col=2,
        )

        # Value loss
        fig.add_trace(
            go.Scatter(x=job_data["iteration"], y=job_data["value_loss"],
                       name=label, line=dict(color=color), legendgroup=label,
                       showlegend=False),
            row=2, col=1,
        )

        # Reward distribution (box plot)
        fig.add_trace(
            go.Box(y=job_data["reward_mean"], name=label,
                   marker_color=color, legendgroup=label, showlegend=False),
            row=2, col=2,
        )

    fig.update_layout(
        title_text=f"Experiment Comparison: {environment}",
        height=800,
        template="plotly_white",
    )
    fig.show()
else:
    print("No experiment data available to compare.")

## Summary Statistics

In [ ]:
if len(df_curves) > 0:
    summary = df_curves.groupby("job_id").agg(
        iterations=("iteration", "max"),
        mean_reward=("reward_mean", "mean"),
        max_reward=("reward_mean", "max"),
        final_reward=("reward_mean", "last"),
        mean_policy_loss=("policy_loss", "mean"),
        mean_value_loss=("value_loss", "mean"),
    ).round(4)

    print("Experiment Summary:")
    summary
else:
    print("No data available for summary.")